# Exploration 05 — new formulas, error structure, and what the models use

> **Exploratory** (CLAUDE.md, "Exploration"). **Inputs:** headline-legal feature sets (Z and R only). **Data:** the
> training sets S_(s,1000) only. No test set is read.

**Part 5 of the classical analysis.** Uses the features (02), the scaling and target transform (03) and the
learning-curve results (04; run 04 first).

The brief asks whether a model works on **formulas absent from training**. The unseen-formula test set answers that
once, at the end (M7). Here a **formula-grouped cross-validation** inside the training set estimates it without
touching that test set: no formula in a validation fold appears in that fold's training part, as for the unseen
test set. Comparing it with ordinary random folds (notebook 04) shows how much each model relies on having seen
the formula. The 8-component variants are included because the quantum model will face this regime.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from qm9dipole import REPO_ROOT
from qm9dipole.analysis import error_by_group, summarize_curves
from qm9dipole.evaluate import group_kfold, kfold, learning_curve, permutation_importance_cv, rmse, tune
from qm9dipole.explore import feature_sets, load_catalog, load_features, load_preprocessing
from qm9dipole.models import classical
from qm9dipole.plots import AXIS, INK_SECONDARY, MODEL_COLORS, MODEL_LABELS, SEQUENTIAL, SERIES, SURFACE, use_style
from qm9dipole.provenance import RESULTS_DIR, save_result
from qm9dipole.splits import load_splits

use_style()
FIG_DIR = REPO_ROOT / "figures"
RANDOM_CV = RESULTS_DIR / "explore04_classical_cv.csv"
if not RANDOM_CV.exists():
    raise FileNotFoundError("results/explore04_classical_cv.csv not found: run explore_04_classical_models.ipynb first")
features = load_features()
SETS = feature_sets(load_catalog())
PRE = load_preprocessing()
SCALING, TARGET = PRE["scaling"], PRE["target"]
splits = load_splits()
y = features["mu"]
formula = features["formula"]
N = 1000
MODELS = ("mean", "ridge", "rbf_krr", "rf", "xgb")
LEGAL_FULL = ["composition", "cm", "structure", "cm+structure"]
train_n = {s: {N: by_n[N]} for s, by_n in splits.train.items()}
print(f"scaling = {SCALING}, target = {TARGET} (notebook 03); S_(s,{N}) holds "
      + ", ".join(str(formula.loc[t[N]].nunique()) for t in train_n.values()) + " formulas (seeds 0, 1, 2)")

## 1. Familiar vs new formulas: random folds vs formula-grouped folds

Same training sets S_(s,1000), models, grids and preprocessing as notebook 04. Only the folds change, to
`GroupKFold(5, shuffle=True, random_state=s)` grouped by formula. Runtime: about 10 minutes.

In [ ]:
def formula_folds(ids, seed):
    return group_kfold(formula.loc[ids].to_numpy(), seed)


frames = {name: features[SETS[name]] for name in LEGAL_FULL}
frames |= {f"{name}/pca8": features[SETS[name]] for name in ("cm", "cm+structure")}


def make_specs(seed, fname):
    models = ("ridge", "rbf_krr") if fname.endswith("/pca8") else MODELS
    return classical.specs(seed, compressed=fname.endswith("/pca8"), models=models, scaling=SCALING, target=TARGET)


grouped, _ = learning_curve(make_specs, frames, y, train_n, make_folds=formula_folds, eval_set="cv_by_formula")
grouped.insert(0, "representation", grouped["features"].str.removesuffix("/pca8"))
grouped.insert(1, "input_variant", np.where(grouped["features"].str.endswith("/pca8"), "compressed", "full"))
save_result(grouped, "explore05_cv_by_formula", n_train=N, seeds=list(train_n), scaling=SCALING, target=TARGET,
            folds="GroupKFold(5, shuffle=True, random_state=seed) by formula",
            label="exploratory; legal inputs; formula-grouped CV inside S_(s,1000) only; no test set read")

random_cv = pd.read_csv(RANDOM_CV)
random_cv = random_cv[(random_cv["n_train"] == N) & random_cv["features"].isin(frames)]
both = pd.concat([random_cv, grouped], ignore_index=True)
gap = summarize_curves(both, by=("model", "features", "eval_set"))
wide = gap.pivot_table(index=["features", "model"], columns="eval_set", values="mean").dropna()
wide["new-formula penalty (D)"] = wide["cv_by_formula"] - wide["cv"]
wide = wide.rename(columns={"cv": "random folds", "cv_by_formula": "formula-grouped folds"})
display(wide.style.format("{:.3f}").set_caption(f"CV MAE (D) at N = {N}, mean over 3 seeds"))

In [ ]:
rows = [(f, m) for f, m in wide.index if m != "mean"]
fig, ax = plt.subplots(figsize=(8.5, 0.32 * len(rows) + 1.3))
for i, (f, m) in enumerate(rows):
    a, b = wide.loc[(f, m), "random folds"], wide.loc[(f, m), "formula-grouped folds"]
    ax.plot([a, b], [i, i], color=MODEL_COLORS[m], linewidth=1.75)
    ax.scatter([a], [i], color=MODEL_COLORS[m], s=36, zorder=3, edgecolors=SURFACE, linewidths=1)
    ax.scatter([b], [i], facecolors=SURFACE, edgecolors=MODEL_COLORS[m], s=36, zorder=3, linewidths=1.5)
ax.set(yticks=range(len(rows)), yticklabels=[f"{f} · {MODEL_LABELS[m]}" for f, m in rows],
       xlabel=f"CV MAE (D), N = {N}, mean over 3 seeds",
       title="Filled: random folds (familiar formulas)   Hollow: formula-grouped folds (new formulas)")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
fig.savefig(FIG_DIR / "explore05_new_formula_gap.png")
plt.show()

## 2. Where do the errors come from?

For the two strongest full legal models at N = 1000 in notebook 04 (each on its best legal set), the out-of-fold
predictions on S_(0,1000) are recomputed with the same folds and search. The errors are then broken down by the
size of the true |μ|.

In [ ]:
legal = random_cv[random_cv["features"].isin(LEGAL_FULL) & (random_cv["model"] != "mean")]
s1000 = summarize_curves(legal)
top = s1000.sort_values("mean").drop_duplicates("model").head(2)
display(top.assign(model=top["model"].map(MODEL_LABELS)).style.format({"mean": "{:.3f}", "std": "{:.3f}"}).hide(axis="index"))

SEED = 0
ids = splits.train[SEED][N]
ys = y.loc[ids]
folds = kfold(len(ids), SEED)
oof, fitted = {}, {}
for r in top.itertuples():
    X = features.loc[ids, SETS[r.features]].to_numpy()
    res = tune(*classical.build(r.model, SEED, scaling=SCALING, target=TARGET), X, ys.to_numpy(), folds)
    oof[r.model] = pd.Series(res.oof, index=ids)
    fitted[r.model] = (r.features, res.params)
    print(f"{MODEL_LABELS[r.model]} on {r.features}: CV MAE {res.mae:.3f} D, RMSE {res.rmse:.3f} D")

In [ ]:
fig, axes = plt.subplots(1, len(oof), figsize=(5.2 * len(oof), 4.4), sharey=True)
lim = (0, float(np.ceil(max(ys.max(), max(p.max() for p in oof.values())))))
for ax, (m, pred) in zip(np.atleast_1d(axes), oof.items()):
    hb = ax.hexbin(ys, pred, gridsize=40, bins="log", cmap=SEQUENTIAL, mincnt=1, linewidths=0, extent=(*lim, *lim))
    ax.plot(lim, lim, color=INK_SECONDARY, linewidth=1)
    ax.set(xlim=lim, ylim=lim, xlabel="true |μ| (D)", title=f"{MODEL_LABELS[m]} ({fitted[m][0]})", aspect="equal")
np.atleast_1d(axes)[0].set_ylabel("out-of-fold prediction (D)")
fig.colorbar(hb, ax=axes, label="molecules (log scale)", shrink=0.85, pad=0.02)
fig.savefig(FIG_DIR / "explore05_parity.png")
plt.show()

bands = pd.cut(ys, [0, 1, 2, 3, 4, 6, 9, np.inf], right=False, labels=["0–1", "1–2", "2–3", "3–4", "4–6", "6–9", "≥ 9"])
by_band = pd.concat({MODEL_LABELS[m]: error_by_group(ys, p, bands).set_index("group") for m, p in oof.items()}, axis=1)
display(by_band.style.format("{:.3f}", subset=[c for c in by_band.columns if c[1] != "n"])
        .set_caption("Out-of-fold errors by true |μ| band, S_(0,1000). Bias = mean(prediction − truth)"))
tail = ys >= 9
for m, p in oof.items():
    print(f"{MODEL_LABELS[m]}: RMSE {rmse(ys, p):.3f} D on all {len(ys)}; {rmse(ys[~tail], p[~tail]):.3f} D without "
          f"the {tail.sum()} molecules at ≥ 9 D (MAE {np.abs(ys - p).mean():.3f} → {np.abs(ys - p)[~tail].mean():.3f})")

## 3. What do the models use?

**Permutation importance:** shuffle one feature column in the validation fold and measure how much the MAE rises.
Each fold's model is fit on its training part and scored on molecules it hasn't seen. Correlated features share
credit (shuffling one leaves its correlates intact), so groups of similar features, such as neighbouring Coulomb
eigenvalues or n_H and the C–H bond count, can look individually weak.

In [ ]:
imp_model = "xgb"
results_imp = {}
for rep in ("cm+structure", "structure"):
    cols = SETS[rep]
    X = features.loc[ids, cols].to_numpy()
    est, grid = classical.build(imp_model, SEED, scaling=SCALING, target=TARGET)
    params = tune(est, grid, X, ys.to_numpy(), folds).params
    imp = permutation_importance_cv(est.set_params(**params), X, ys.to_numpy(), folds, np.random.default_rng(SEED), n_repeats=3)
    results_imp[rep] = pd.Series(imp, index=cols).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5), gridspec_kw={"wspace": 0.6})
for ax, (rep, imp) in zip(axes, results_imp.items()):
    show = imp.tail(15)
    ax.barh(show.index, show.values, color=SERIES[0], height=0.68)
    ax.axvline(0, color=AXIS, linewidth=0.8)
    ax.set(xlabel="rise in validation MAE when shuffled (D)", title=f"{MODEL_LABELS[imp_model]} on {rep}: top 15")
    ax.grid(axis="y", visible=False)
fig.savefig(FIG_DIR / "explore05_permutation_importance.png")
plt.show()
_ = save_result(pd.concat({rep: s.rename("mae_rise_D") for rep, s in results_imp.items()}, names=["features", "feature"])
                .reset_index(), "explore05_permutation_importance", model=imp_model, seed=SEED, n_train=N, n_repeats=3,
            label="exploratory; legal inputs; 5-fold CV inside S_(0,1000)")

## Summary

1. **New formulas cost little at N = 1000.** Grouping the folds by formula raises CV MAE by at most 0.06 D, and by
   only about 0.01 D for the structure features. Each training set already spans about 360 formulas, so most
   chemistry is covered. The Coulomb spectrum with kernel ridge degrades most (+0.058 D, and +0.051 D for its
   8-component version), so the representation the quantum model will use is the one most sensitive to unseen
   formulas. The unseen-formula test set (M7) will give the definitive answer.
2. **The best classical models** are RBF kernel ridge (0.713 D) and XGBoost (0.716 D) on the 31 structure features.
   Adding the Coulomb spectrum does not help once the structure features are present.
3. **Errors regress toward the mean.** Molecules below 1 D are over-predicted (bias ≈ +0.5 D) and those above 3 D
   under-predicted. The 9 zwitterion-like molecules at ≥ 9 D are missed almost entirely (bias ≈ −9 D), and they
   raise RMSE from 0.98 to 1.41 D. No structure-only feature here identifies a zwitterion. An explicit
   charged-group feature (protonated amine plus carboxylate) is a candidate for future feature engineering.
4. **What the models use:** the engineered dipole estimates (`en_point_dipole`, `bond_dipole_norm`), then nitrile
   and carbonyl counts read from bond lengths, the heteroatom offset, N–H bonds and fluorine. That is chemistry: the
   polar groups and how their bond dipoles add up.